# RL Test Flow Optimization — Notebook 2 of 3: Stage-2 + HPO

**Prerequisite**: NB1 complete + `rl-stage1-results` dataset added as input.
**RL training device**: CPU (intentional for SB3 MLP workloads).

| Step | Description | Est. Time (Kaggle T4 CPU) |
|------|-------------|---------------------------|
| 0 | Reinstall + load NB1 results | ~3 min |
| 6 | Stage-2: top-2 algos × 3 seeds × 500K steps | ~60-70 min |
| 7 | Optuna HPO: 30 trials × 100K steps on best algo | ~55-65 min |
| Save | Serialize everything to `/kaggle/working/rl_stage2/` | ~1 min |

**Total: ~2-2.5 hours**  (well within Kaggle's 12-hour limit)

## Timing Reference (from NB1 on this Kaggle session)
| Algorithm | 200K steps | 100K steps | 500K steps |
|-----------|-----------|------------|------------|
| DQN (best) | 7.4 min | ~3.7 min | ~18.5 min |
| PPO (2nd) | 7.1 min | ~3.6 min | ~17.8 min |

> **After NB2 completes**: Output tab → Create Dataset → name it **`rl-stage2-results`**
> Then open NB3 and add both `rl-stage1-results` AND `rl-stage2-results` as inputs.

## Evaluation Fix (applied in this NB)
NB1 revealed that post-training evaluation was broken: a deterministic A2C/PPO policy
loops on duplicate actions, accumulating -1×410 = -410 reward/episode. Fixed in `agent.py`:
- MaskablePPO: passes `action_masks` to `model.predict()` (the correct sb3-contrib API)
- PPO/DQN/A2C: if predicted action is invalid (duplicate/unaffordable), auto-STOP instead

## Step 0: Reinstall + Load NB1 Results

In [ ]:
import subprocess, sys, os, json, time as _time
subprocess.run(['pip', 'install', '-q',
    'stable-baselines3[extra]', 'sb3-contrib', 'gymnasium',
    'optuna', 'mlflow', 'pyarrow', 'matplotlib', 'numpy', 'torch'], check=True)

!rm -rf rl-test-flow-optimization
!git clone https://github.com/rajendarmuddasani/rl-test-flow-optimization.git
os.chdir('rl-test-flow-optimization')
sys.path.insert(0, '.')

# Verify evaluation fix is present
import inspect
from src.agent import evaluate_trained_model
assert 'is_maskable' in inspect.getsource(evaluate_trained_model), \
    'ERROR: Old evaluation code without action-mask fix! Check git clone.'
print('Code version verified: action-mask evaluation fix present ✓')

import torch, numpy as np, pandas as pd
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
print(f'GPU: {gpu}  |  RL train device: cpu (forced for SB3 MlpPolicy)')

# Load NB1 results
NB1_PATH = '/kaggle/input/rl-stage1-results/stage1_results.json'
with open(NB1_PATH) as f:
    nb1 = json.load(f)

baseline_results = nb1['baselines']
stage1_results   = nb1['stage1']
top2_algos       = nb1['top2_algos']
best_bl          = nb1['best_base_reward']
baseline_df      = pd.DataFrame(baseline_results).T
stage1_df        = pd.DataFrame(stage1_results).T

print(f'\nNB1 results loaded successfully:')
print(f'  Top-2 algos for Stage-2: {top2_algos}')
print(f'  Best baseline reward: {best_bl:+.2f} (cost_efficient heuristic)')
print(f'  Stage-1 winner: {stage1_df["mean_reward"].idxmax()} '
      f'= {stage1_df["mean_reward"].max():+.2f}')
print('\nStage-1 table:')
print(stage1_df[['mean_reward','accuracy','mean_cost','mean_tests']].to_string())


## Step 6: Stage-2 — Deep Training Top-2 (500K steps × 3 seeds)

3 random seeds quantify variance — RL optimisation is stochastic.
A single run could be lucky or unlucky; mean ± std across seeds is the production-grade metric.

| What we measure | Why it matters |
|-----------------|----------------|
| Mean reward | Raw RL performance |
| Std across seeds | Reproducibility — low std = stable policy |
| Accuracy | % chips correctly classified (defect vs clean) |
| Mean cost/chip | AMD production metric — lower = cheaper test run |

Expected: each algo × seed run ≈ 18-19 min. Total ≈ 60-65 min.

In [ ]:
import sys
from src.environment import TestFlowEnv
from src.agent import ALGO_REGISTRY, evaluate_trained_model
from pathlib import Path

def fp(*a, **k): print(*a, **k); sys.stdout.flush()

STAGE2_STEPS = 500_000
SEEDS        = [42, 123, 777]
SAVE_DIR2    = Path('/kaggle/working/rl_stage2')
SAVE_DIR2.mkdir(parents=True, exist_ok=True)

# Copy NB1 stage1_results.json into working dir
import shutil
shutil.copy(NB1_PATH, SAVE_DIR2 / 'stage1_results.json')

stage2_results = {}
total_runs = len(top2_algos) * len(SEEDS)
run_idx = 0
overall_t0 = _time.time()

fp(f'Stage-2: {STAGE2_STEPS:,} steps × {len(SEEDS)} seeds × {len(top2_algos)} algos')
fp(f'Total runs: {total_runs}  |  Est. ~{total_runs * 18:.0f} min total')
fp('=' * 70)

for algo_name in top2_algos:
    train_fn = ALGO_REGISTRY[algo_name]
    seed_metrics = []
    for seed in SEEDS:
        run_idx += 1
        elapsed_so_far = (_time.time() - overall_t0) / 60
        fp(f'\n[Run {run_idx}/{total_runs}] {algo_name.upper()} seed={seed}')
        fp(f'  Start: {_time.strftime("%H:%M:%S")}  '
           f'| elapsed so far: {elapsed_so_far:.1f}m  '
           f'| est. remaining: {(total_runs - run_idx + 1) * 18:.0f}m')
        env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)
        t0 = _time.time()
        model = train_fn(
            env,
            total_timesteps=STAGE2_STEPS,
            output_dir=str(SAVE_DIR2 / f'{algo_name}_seed{seed}'),
            seed=seed,
        )
        elapsed = _time.time() - t0
        fp(f'  Training done in {elapsed/60:.1f}m. Evaluating 100 episodes...')
        metrics = evaluate_trained_model(env, model, n_episodes=100)
        metrics['seed']           = seed
        metrics['train_time_sec'] = round(elapsed, 1)
        seed_metrics.append(metrics)
        fp(f'  DONE: reward={metrics["mean_reward"]:+.2f} | '
           f'acc={metrics["accuracy"]:.3f} | '
           f'cost={metrics["mean_cost"]:.2f} | '
           f'tests={metrics["mean_tests"]:.1f}')
    stage2_results[algo_name] = seed_metrics

fp(f'\n{"="*70}')
fp(f'STAGE-2 COMPLETE  |  Total: {(_time.time()-overall_t0)/60:.1f} min')
fp(f'\n{"Algorithm":20s} {"Mean Reward":>12s} {"Std±":>8s} {"Accuracy":>10s} {"Cost":>8s}')
fp('-' * 70)
for algo in top2_algos:
    rewards = [m['mean_reward'] for m in stage2_results[algo]]
    accs    = [m['accuracy']    for m in stage2_results[algo]]
    costs   = [m['mean_cost']   for m in stage2_results[algo]]
    fp(f'{algo:20s} {np.mean(rewards):>+12.2f} {np.std(rewards):>8.3f} '
       f'{np.mean(accs):>10.3f} {np.mean(costs):>8.2f}')
fp(f'\nBaseline best: {best_bl:+.2f} (cost_efficient heuristic)')


## Step 7: Optuna HPO — 30 Trials × 100K Steps

Tree-structured Parzen Estimator (TPE) searches for the best hyperparameters.

| Hyperparameter | Search Space | Why |
|----------------|-------------|-----|
| `learning_rate` | 1e-5 → 1e-2 (log) | Most impactful RL param |
| `gamma` | 0.90 → 0.999 | Discount: near-sighted vs far-sighted |
| `batch_size` | 32, 64, 128, 256 | Gradient noise vs compute |

**Algorithm**: `top2_algos[0]` (DQN from NB1)  
**30 trials × ~3.7 min each ≈ 55-65 min**  
AMD production standard: 30+ trials minimum for reliable HPO.

In [ ]:
import sys
from src.agent import run_optuna_hpo

def fp(*a, **k): print(*a, **k); sys.stdout.flush()

best_algo = top2_algos[0]  # DQN from NB1
n_trials  = 30
hpo_steps = 100_000

fp(f'Optuna HPO: algo={best_algo}, trials={n_trials}, steps/trial={hpo_steps:,}')
fp(f'Est. time: {n_trials} × ~3.7 min = ~{n_trials * 3.7:.0f} min')
fp(f'Started:   {_time.strftime("%H:%M:%S")}')
fp('=' * 60)

t_hpo = _time.time()
hpo_result = run_optuna_hpo(
    env_cls=TestFlowEnv,
    env_kwargs={'n_tests': 200, 'cost_budget': 50.0, 'time_budget': 120.0},
    algo=best_algo,
    n_trials=n_trials,
    timesteps=hpo_steps,
)
hpo_elapsed = (_time.time() - t_hpo) / 60

fp(f'\n{"="*60}')
fp(f'OPTUNA COMPLETE  |  {hpo_elapsed:.1f} min')
fp(f'  Best reward: {hpo_result["best_value"]:+.2f}')
fp(f'  Best params:')
for k, v in hpo_result['best_params'].items():
    fp(f'    {k}: {v}')
fp(f'  Total trials completed: {hpo_result["n_trials"]}')


## Save NB2 Outputs

Serialises all stage-2 + HPO results plus copies of stage-1 data.

> **After this cell completes**: Output tab → Create Dataset → **`rl-stage2-results`**
> 
> Then open NB3 and add BOTH:
> - `rl-stage1-results` (from NB1)
> - `rl-stage2-results` (from this notebook)

In [ ]:
import json, sys
def fp(*a, **k): print(*a, **k); sys.stdout.flush()

save_data2 = {
    'baselines':        baseline_results,
    'stage1':           stage1_results,
    'stage2':           stage2_results,
    'hpo':              hpo_result,
    'top2_algos':       top2_algos,
    'best_algo':        best_algo,
    'best_base_reward': float(best_bl),
    'best_params':      hpo_result['best_params'],
}

with open(SAVE_DIR2 / 'stage2_results.json', 'w') as f:
    json.dump(save_data2, f, indent=2, default=str)

fp('=== NB2 ARTIFACTS ===')
total = 0
for ff in sorted(SAVE_DIR2.rglob('*')):
    if ff.is_file() and ff.suffix in ['.json', '.zip']:
        fp(f'  {str(ff.relative_to(SAVE_DIR2)):60s} {ff.stat().st_size/1e3:6.0f} KB')
        total += ff.stat().st_size
fp(f'\nTotal: {total/1e6:.1f} MB')
fp('\nNB2 complete!')
fp('Next: Output tab → Create Dataset → rl-stage2-results')
fp('Then open NB3 and add both rl-stage1-results + rl-stage2-results as inputs')
